# Training API

Fine-tune forecasting models on your Lightning Rod datasets. This notebook walks through the full training workflow: generating a dataset, estimating cost, creating a training job, and monitoring progress.

The training API supports LoRA fine-tuning with configurable base models, training steps, batch size, and rank.

## Install the SDK

In [1]:
%pip install lightningrod-ai python-dotenv

from IPython.display import clear_output
clear_output()

## Set up the client

Sign up at [dashboard.lightningrod.ai](https://dashboard.lightningrod.ai/?redirect=/api) to get your API key and **$50 of free credits**.

- **Google Colab**: Go to the Secrets section (key icon in left sidebar) and add a secret named `LIGHTNINGROD_API_KEY`
- **Local Jupyter**: Set the `LIGHTNINGROD_API_KEY` environment variable, or you'll be prompted to enter it

In [2]:
from dotenv import load_dotenv
from lightningrod import LightningRod
from lightningrod.utils import config

load_dotenv()
api_key = config.get_config_value("LIGHTNINGROD_API_KEY")

lr = LightningRod(api_key=api_key)

## Get a dataset ID

Training requires a dataset ID from a pipeline run. Run one of the other notebooks first to generate a dataset - each one prints the **Dataset ID** after `transforms.run()` — copy it into the cell below.

In [4]:
default_dataset_id = "a2119549-25e6-4deb-87b2-8164949cdb61" # paste it here, or set it as an environment variable
dataset_id = config.get_config_value("DATASET_ID", default_dataset_id)

## Estimate training cost

Before starting a job, use `estimate_cost` to see the expected cost and token usage.

In [4]:
from lightningrod.training import TrainingConfig

config = TrainingConfig(
    input_dataset_id=dataset_id,
    base_model="Qwen/Qwen3-4B-Instruct-2507",
    training_steps=10,
)

cost_estimate = lr.training.estimate_cost(config)
print(f"Estimated cost: ${cost_estimate.total_cost_dollars:.2f}")
print(f"Effective steps: {cost_estimate.effective_steps}")
print(f"Train tokens: {cost_estimate.train_tokens:,}")
print(f"Notes: {cost_estimate.notes}")

Estimated cost: $4.53
Effective steps: 10
Train tokens: 10,082,082
Notes: Estimate uses 0.5× max_response_length for output; actual may vary


## Start training

`run` creates a job and polls until completion with a live progress display. Use this when you want to wait for the job to finish in the notebook. Skip this if you used `create` above and prefer to poll manually.

In [ ]:
job = lr.training.run(config, name="Forecasting fine-tune", poll_interval=15)
print(f"Job {job.id} completed with status: {job.status}")
print(f"Trained model ID: {job.model_id}")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Training RUNNING                                                                                            │
│                                                                                                                 │
│    Job: Forecasting fine-tune                                                                                   │
│                                                                                                                 │
│    Progress: [██░░░░░░░░░░░░░░░░░░░░░░] 1/10 (10%)                                                              │
│                                                                                                                 │
│    Reward: latest -1.5703  avg -1.5703  (1 steps)  (higher is better)                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## List and get jobs

List all training jobs or fetch a specific job by ID.

In [11]:
import pandas as pd

jobs_response = lr.training.list(limit=5)

df = pd.DataFrame([
    {
        "Job ID": j.id,
        "Status": j.status,
        "Base Model": getattr(j.config, "base_model", None),
        "Trained Model ID": j.model_id,
    }
    for j in jobs_response.jobs
])

df

,Job ID,Status,Base Model,Trained Model ID
0,92e583d7-6b86-4aa2-85cf-f2802d72dc62,COMPLETED,Qwen/Qwen3-4B-Instruct-2507,checkpoint:92e583d7-6b86-4aa2-85cf-f2802d72dc62
1,dcfcb83a-c091-4af1-8a23-79da1e3537ce,RUNNING,Qwen/Qwen3-4B-Instruct-2507,None
2,e0813bbe-0970-4102-918e-653ad68a1026,FAILED,qwen/Qwen3-4B-Instruct-2507,None
3,b84af684-7eb0-4ebf-976e-9e0a91d01466,FAILED,qwen/Qwen3-4B-Instruct-2507,None
4,2bf4a66f-eb34-409e-b36c-f3b5d96db141,STARTING,qwen/Qwen3-4B-Instruct-2507,None


## Inference with your trained model

Once training completes, use `job.model_id` with the OpenAI-compatible API. We also have a pre-trained foresight-v3 model for forecasting — see [08_foresight_model.ipynb](08_foresight_model.ipynb).

In [37]:
%pip install openai
from IPython.display import clear_output
clear_output()

from openai import OpenAI
from lightningrod.utils import config

base_url = config.get_config_value("LIGHTNINGROD_BASE_URL", "https://api.lightningrod.ai/api/public/v1")
client = OpenAI(api_key=api_key, base_url=f"{base_url}/openai")

In [39]:
response = client.chat.completions.create(
    model="checkpoint:92e583d7-6b86-4aa2-85cf-f2802d72dc62",
    messages=[
        {"role": "system", "content": "Answer as a probability between 0 and 1 between <answer></answer> tags."},
        {"role": "user", "content": "Will the Fed cut rates by 25bp in March 2026?"}
    ]
)
print(response.choices[0].message.content)

<answer>0.15</answer>


## Run evals on trained model

Run test evals on your trained model against a test dataset. The eval job runs the model on the dataset and reports metrics. Use the same dataset for a quick check, or a separate test split for production.

In [6]:
eval_job = lr.evals.run(
    model_id="checkpoint:92e583d7-6b86-4aa2-85cf-f2802d72dc62",
    test_dataset_id=dataset_id,
)
print(f"Eval {eval_job.id} completed with status: {eval_job.status}")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Eval COMPLETED                                                                                              │
│                                                                                                                 │
│    Model: checkpoint:92e583d7-6b86-4aa2-85cf-f2802d72dc62                                                       │
│    Test dataset: a2119549-25e6-4deb-87b2-8164949cdb61                                                           │
│                                                                                                                 │
│    base: {'ece': 0.781211711711727, 'n_valid': 1998, 'n_samples': 2000, 'parse_rate': 0.999, 'brier_score':     │
│  0.7231255260260259, 'mean_reward': -2.230670151721925, 'mean_valid_reward': -2.2248950467686943}               │
│    trained: {'ece': 0.07779999999999312, 'n_valid': 2000, 'n_samples': 2000, 'parse_rate': 1.0, 'brier_score':  │
│  0.11209000000000004, 'mean_reward': -0.3894061055473225, 'mean_valid_reward': -0.3894061055473225}             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Eval c9da64dd-3826-44fd-9f68-17920f4feeb9 completed with status: COMPLETED


In [10]:
import pandas as pd

evals_response = lr.evals.list(limit=5)
pd.DataFrame([
    {
        "Eval ID": e.id,
        "Status": e.status,
        "Test Dataset": e.test_dataset_id,
        "Metrics": dict(e.metrics.additional_properties) if hasattr(e.metrics, "additional_properties") else None,
    }
    for e in evals_response.jobs
])

,Eval ID,Status,Test Dataset,Metrics
0,c9da64dd-3826-44fd-9f68-17920f4feeb9,COMPLETED,a2119549-25e6-4deb-87b2-8164949cdb61,"{'base': {'ece': 0.781211711711727, 'n_valid':..."


> Note: the trained model checkpoint will only be available for the period of 7 days. If you wish to host this model long-term, reach out to us at support@lightningrod.ai.